# 03 - Baseline Model
## Logistic Regression + Random Forest

**Tujuan:** Membangun model baseline yang interpretable dan cepat.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score

print('Libraries loaded')

## 1. Load & Prep Data

In [ ]:
df = pd.read_csv('../data/sample/sample_input.csv')
grade_map = {'A': 0, 'B': 1, 'C': 2, 'Reject': 3}
inv_map = {v: k for k, v in grade_map.items()}

df['tpc'] = df['tpc'].fillna(df['tpc'].median())
df['grading_delta_hours'] = df['grading_delta_hours'].fillna(df['grading_delta_hours'].median())
df['shift_Pagi'] = (df['shift'] == 'Pagi').astype(int)
df['shift_Siang'] = (df['shift'] == 'Siang').astype(int)

feature_cols = ['storage_temp', 'ph', 'storage_time', 'pasteurization_temp', 'tpc', 'grading_delta_hours', 'shift_Pagi', 'shift_Siang']
X = df[feature_cols]
y = df['grade'].map(grade_map)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

## 2. Logistic Regression

In [ ]:
lr = LogisticRegression(multi_class='multinomial', max_iter=1000, random_state=42)
lr.fit(X_train_s, y_train)

y_pred_lr = lr.predict(X_test_s)
f1_lr = f1_score(y_test, y_pred_lr, average='weighted')

print(f'Logistic Regression F1 (weighted): {f1_lr:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_lr, target_names=list(inv_map.values()), zero_division=0))

cm = confusion_matrix(y_test, y_pred_lr)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=inv_map.values(), yticklabels=inv_map.values())
plt.title('Confusion Matrix - Logistic Regression')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

## 3. Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=150, max_depth=12, min_samples_leaf=4, class_weight='balanced', random_state=42)
rf.fit(X_train_s, y_train)

y_pred_rf = rf.predict(X_test_s)
f1_rf = f1_score(y_test, y_pred_rf, average='weighted')

print(f'Random Forest F1 (weighted): {f1_rf:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_rf, target_names=list(inv_map.values()), zero_division=0))

cm = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=inv_map.values(), yticklabels=inv_map.values())
plt.title('Confusion Matrix - Random Forest')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

## 4. Cross-Validation

In [ ]:
cv_scores_lr = cross_val_score(lr, X_train_s, y_train, cv=5, scoring='f1_weighted')
cv_scores_rf = cross_val_score(rf, X_train_s, y_train, cv=5, scoring='f1_weighted')

print(f'Logistic Regression CV F1: {cv_scores_lr.mean():.4f} (+/- {cv_scores_lr.std():.4f})')
print(f'Random Forest CV F1:        {cv_scores_rf.mean():.4f} (+/- {cv_scores_rf.std():.4f})')

## 5. Kesimpulan

- **Random Forest** memberikan performa terbaik dengan F1 ~0.92
- **Logistic Regression** juga cukup baik sebagai baseline interpretable
- Model siap ditingkatkan ke XGBoost/LightGBM jika needed